<a href="https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/daniausman24-bot/ML_Internship_Track"
REPO_DIR = "ML_Internship_Track"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ML_Internship_Track/ML_Internship_Track
Starter data found. You're ready.


Lane: Refresh / Content Opportunity Scoring.

54% of the 30,000 pages in the starter dataset are flagged as declining
(trend_direction == "down") — this isn't a rare edge case, it's the majority
of the content FlyRank's clients own. With 32 clients and no way to hand-review
every page, someone has to decide which declining pages get an editor's
attention first. That's a ranking problem, and it's why this lane over the
others: the baseline pipeline in this repo already targets this exact label,
so I can build on solid ground and spend my time on the ranking logic itself
rather than reinventing the data contract.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df["trend_direction"].value_counts())
print("pct declining:", round((df["trend_direction"] == "down").mean() * 100, 1))

(30000, 44)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
pct declining: 54.2


Decision: given a client's declining pages, which one should the content
editor refresh this week?

Who acts: a FlyRank content editor working through a client's refresh queue.
They pick pages off my ranked list and rewrite/update them.

Cost of a wrong call: declining pages in this dataset account for roughly
217,000 of the 90-day clicks in the sample — comparable to the ~204,000 clicks
still going to stable pages. Sending an editor to refresh a page that isn't
actually losing meaningful traffic wastes their time and delays a fix on a
page that IS bleeding clicks. The cost is misallocated editor hours, not a
catastrophic error — which is why a ranked queue, not a single yes/no call,
is the right shape for this.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
clicks_by_trend = df.groupby("trend_direction")["clicks_90d"].sum()
print(clicks_by_trend)

trend_direction
down      217248
flat          64
new          863
stable    203557
up         61188
Name: clicks_90d, dtype: int64


- 16,262 of 30,000 pages (54.2%) are declining.
- Declining pages sit at an average content age of 236 days vs. 288-295 days
  for pages that are up or stable — they're not just old content aging out,
  something else is going on.
- Declining pages have gone a bit longer without an update (avg 49 days since
  last update) than pages trending up (avg 43 days) — a small gap, worth
  digging into further rather than treating as proof on its own.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.groupby("trend_direction")[["content_age_days", "days_since_last_update"]].mean())

                 content_age_days  days_since_last_update
trend_direction                                          
down                   236.178637               49.245788
flat                   245.889757               37.453993
new                    238.718247               22.337209
stable                 295.439953               50.061892
up                     288.478806               43.425706


## 4. Careful words: what I can and can't claim

I can say: which pages are observed to be declining, and which signals are
directionally associated with decline in this sample. My output will be
decision-support — a ranked list an editor can choose to act on.

I cannot say: that any signal here causes the decline, or that refreshing a
page will reverse it — that needs a before/after test, not a snapshot. And I
will never claim to be predicting Google's ranking algorithm; trend_direction
and trend_pct describe what already happened to this page, not how search
ranking works in general. They're also excluded as model features later,
since the label is derived from them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = {"trend_direction", "trend_pct"}
print("excluded from features:", excluded)

excluded from features: {'trend_pct', 'trend_direction'}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.